In [139]:
import pathlib as pl

cfg_nb = pl.Path("../../../load-config.ipynb").resolve(strict=True)
%run $cfg_nb

_NB_SESSION = __session__
NB_NAME = pl.Path(_NB_SESSION).name
NB_PATH = pl.Path(_NB_SESSION).parent
NB_REL_PATH = NB_PATH.relative_to(CONFIG["project_repo"]).joinpath(NB_NAME)

PLOT_ROOT = CONFIG["plot_root"].joinpath(NB_PATH.stem)
PLOT_ROOT.mkdir(exist_ok=True, parents=True)
NB_CACHE_FOLDER = CONFIG["nb_cache_folder"]
DATA_ROOT = CONFIG["project_data"]

# process/plot subregion labelings generated w/ minimap
# workflow-smk-sequence-annotation::10-annotate::regions::minimap::create_annotation_region_db

import pandas as pd
import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import scipy.stats as scistats
import collections as col
from matplotlib.lines import Line2D

label_files = DATA_ROOT.joinpath("wf-data/seqann/region_db").glob("*.bed")

truth_regions = CONFIG["project_repo"].joinpath("annotation", "norm", "HG002-Y_T2Tv2_regions_repeat-disjoin.bed")
truth_regions = pd.read_csv(truth_regions, sep="\t", header=0)
truth_set = list(truth_regions["name"])
truth_set.append("95_uncertain")
truth_lut = dict(
    (row.name, float(row.size)) for row in truth_regions.itertuples()
)

regions = []
for bedlike in label_files:
    df = pd.read_csv(bedlike, sep="\t", header=0)
    df["sample"] = bedlike.name.split(".")[0]
    df["haplogroup"] = df["sample"].apply(lambda s: s.split("-")[-1])
    df["size"] = df["end"] - df["start"]
    df["size_log10"] = np.log10(df["size"])
    df["name"] = df["name"].apply(lambda n: n.split("_", 2)[-1])
    regions.append(df)

regions = pd.concat(regions, axis=0, ignore_index=False)
regions.drop(["anchor_row", "strand", "score"], axis=1, inplace=True)

# plot by region
size_dist = regions.groupby("name")["size_log10"].apply(np.array)

boxes = []
positions = []
empty = []

for label in truth_set:
    pos = int(label.split("_",1)[0])
    positions.append(pos)
    try:
        if "uncertain" in label:
            values = size_dist["uncertain"]
        else:
            values = size_dist[label]
    except KeyError:
        values = []
        empty.append(pos)
    boxes.append(values)

fig, ax = plt.subplots(figsize=(16,8))
boxplot = ax.boxplot(
    boxes,
    positions=positions
)

_ = ax.set_ylim(-0.1, 7.75)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylabel("(split) region size (log10[bp])", fontsize=14)
ax.set_xlabel("Y subregion labels HG002/T2Tv2", fontsize=14)

_ = ax.vlines(empty, 0, 1, ls="dotted", color="red")

_ = ax.set_xticks(positions)
_ = ax.set_xticklabels(truth_set, rotation=90, fontsize=10)

plt.savefig(
    PLOT_ROOT.joinpath("y-region-labels_size-dist.png"),
    dpi=300, bbox_inches="tight"
)
plt.close()

# plot total by sample/region
size_dist = regions.groupby(["sample", "name"])["size"].sum().groupby("name").apply(np.array)

boxes = []
positions = []
empty = []

for label in truth_set:
    pos = int(label.split("_",1)[0])
    positions.append(pos)
    try:
        if "uncertain" in label:
            values = size_dist["uncertain"]
            values.sort()
            median = values[values.size//2]
            values = values / median * 100
        else:
            values = size_dist[label]
            factor = truth_lut[label]
            values = values / factor * 100
    except KeyError:
        values = []
        empty.append(pos)
    boxes.append(values)

fig, ax = plt.subplots(figsize=(16,8))
boxplot = ax.boxplot(
    boxes,
    positions=positions
)

_ = ax.set_ylim(-5, 410)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.set_ylabel("total region size (pct. bp / T2Tv2)", fontsize=14)
ax.set_xlabel("Y subregion labels HG002/T2Tv2", fontsize=14)

_ = ax.vlines(empty, 0, 25, ls="dotted", color="red")

_ = ax.set_xticks(positions)
_ = ax.set_xticklabels(truth_set, rotation=90, fontsize=10)

plt.savefig(
    PLOT_ROOT.joinpath("y-region-labels_totals-by-sample.png"),
    dpi=300, bbox_inches="tight"
)
plt.close()

# plot total uncertain by haplogroup
size_dist = regions.loc[
    regions["name"] == "uncertain",
    ["sample", "haplogroup", "size"]
].groupby(["sample", "haplogroup"])["size"].sum().groupby("haplogroup").apply(np.array)

size_dist /= int(1e6)

positions = np.arange(1, size_dist.size+1, dtype=int)

fig, ax = plt.subplots(figsize=(16,8))
boxplot = ax.boxplot(
    size_dist.values,
    positions=positions
)

_ = ax.set_xticks(positions)
_ = ax.set_xticklabels(
    [f"{i}*" if i == "J1" else f"{i}" for i in size_dist.index],
    fontsize=12
)

_ = ax.set_ylabel("Total region size 'uncertain' per sample (Mbp)", fontsize=14)
_ = ax.set_xlabel("Haplogroup (* = HG002/NA24385)", fontsize=14)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

plt.savefig(
    PLOT_ROOT.joinpath("y-uncertain-regions_total-size-by-hg.png"),
    dpi=300, bbox_inches="tight"
)
plt.close()


# plot scatter/correlation w/ Mark's Yq12 estimates
yq12_est_length_file = CONFIG["project_repo"].joinpath("annotation", "raw", "20250224_HPRC_Lengths_Estimations.ML.csv")
yq12_est_length = pd.read_csv(yq12_est_length_file, sep=",", header=0)

het_only = regions.loc[regions["name"] == "92_HET", :].copy().groupby(["sample", "haplogroup"])["size"].sum().reset_index(drop=False)
het_only["SampleName"] = het_only["sample"].apply(lambda s: s.split("-")[0])
het_only = het_only.merge(yq12_est_length, on="SampleName")
het_only["haploclade"] = het_only["haplogroup"].apply(lambda h: h[0])

clade_colors = mpl.colormaps["tab20"].colors
enum_colors = col.OrderedDict()
for clade, color in zip(sorted(het_only["haploclade"].unique()), clade_colors):
    enum_colors[clade] = color

x_vals = []
y_vals = []
colors = []
for row in het_only.itertuples():
    x_vals.append(row.size/float(1e6))
    y_vals.append(row.KMER_Refined/float(1e6))
    colors.append(enum_colors[row.haploclade])

colors = np.array(colors)

fig, ax = plt.subplots(figsize=(10,10))

ax.scatter(
    x_vals,
    y_vals,
    c=colors,
    s=25
)

ax.set_xlabel("Annotated size HET (gaps open; Mbp)", fontsize=12)
ax.set_ylabel("k-mer estimated size HET (Mbp)", fontsize=12)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

legend_handles = []
for haploclade, color in enum_colors.items():
    legend_handles.append(
        Line2D(
            [], [], marker='o', color=color,
            label=haploclade, ls="none",
            markerfacecolor=color, markersize=5)
    )

ax.legend(handles=legend_handles, loc="best")

spr_corr = scistats.spearmanr(x_vals, y_vals)
ax.text(22.5, 15, f"(spr) r ~ {round(spr_corr[0],2)}", {"fontsize": 16})

ax.set_xlim(5, 30)
ax.set_ylim(5, 45)

ax.plot(
    np.arange(5, 30),
    np.arange(5, 30),
    ls="dotted",
    lw=2,
    zorder=0,
    color="gray"
)

plt.savefig(
    PLOT_ROOT.joinpath("y-het-corr_kmer-annotated.png"),
    dpi=300, bbox_inches="tight"
)
plt.close()